# Bölüm 5: Encoding and Evolution (Kodlama ve Evrim)

> *"Everything changes and nothing stands still."*  
> — Heraclitus of Ephesus (MÖ 360)

Bu notebook, **Designing Data-Intensive Applications** kitabının 5. bölümünü Türkçe olarak açıklamaktadır. Tüm teknik terimler İngilizce korunmuştur.

---

## İçindekiler

1. [Giriş: Evolvability (Evrilebilirlik)](#1)
2. [Formats for Encoding Data (Veri Kodlama Formatları)](#2)
3. [Language-Specific Formats (Dile Özgü Formatlar)](#3)
4. [JSON, XML ve Binary Variants](#4)
5. [JSON Schema](#5)
6. [Binary Encodings (İkili Kodlamalar)](#6)
7. [Protocol Buffers](#7)
8. [Field Tags ve Schema Evolution](#8)
9. [Avro](#9)
10. [The Merits of Schemas (Schema'ların Faydaları)](#10)
11. [Modes of Dataflow (Veri Akışı Modları)](#11)
12. [Dataflow Through Databases](#12)
13. [Dataflow Through Services: REST and RPC](#13)
14. [Durable Execution and Workflows](#14)
15. [Event-Driven Architectures](#15)
16. [Özet (Summary)](#16)

<a id='1'></a>
## 1. Giriş: Evolvability (Evrilebilirlik)

Uygulamalar zamanla kaçınılmaz olarak değişir. Yeni özellikler eklenir, kullanıcı gereksinimleri daha iyi anlaşılır ya da iş koşulları değişir. Bu nedenle sistemleri değişime kolay adapte olabilecek şekilde inşa etmek gerekir; bu özelliğe **evolvability** (evrilebilirlik) denir.

Çoğu zaman bir uygulamanın özelliklerinde yapılan bir değişiklik, depoladığı veride de bir değişiklik gerektirir. Örneğin yeni bir alan (field) ya da kayıt türü (record type) eklenmesi ya da mevcut verinin yeni bir biçimde sunulması gerekebilir.

### Schema Değişiklikleri ve Uyumluluk Zorluğu

**Relational databases** (ilişkisel veritabanları) genellikle tüm verinin tek bir schema'ya uyduğunu varsayar. Bu schema `ALTER` ifadeleriyle değiştirilebilir (schema migration), ancak herhangi bir anda yalnızca bir schema geçerliliğini korur.

**Schema-on-read** ("schemaless") veritabanları ise bir schema zorunluluğu getirmez; bu yüzden veritabanı farklı zamanlarda yazılmış eski ve yeni veri formatlarını bir arada içerebilir.

Veri formatı ya da schema değiştiğinde, uygulama kodunun da değişmesi gerekir. Ancak büyük bir uygulamada kod değişiklikleri anlık olarak gerçekleşemez:

- **Server-side applications** için **rolling upgrade** (aşamalı güncelleme) yapılmak istenebilir: Yeni sürüm önce birkaç node'a dağıtılır, düzgün çalışıp çalışmadığı izlenir, ardından yavaş yavaş tüm node'lara yayılır. Bu yaklaşım servis kesintisi olmaksızın yeni sürüm dağıtımına olanak sağlar.

- **Client-side applications** söz konusu olduğunda ise güncellemeyi yükleyip yüklemeyeceğine kullanıcı karar verir ve bu süreci kontrol edemezsiniz.

Bu durum şunu ifade eder: Sistemde aynı anda eski ve yeni kod sürümleri ile eski ve yeni veri formatları bir arada bulunabilir.

### Backward Compatibility ve Forward Compatibility

Sistemin sorunsuz çalışmaya devam etmesi için iki yönlü **compatibility** (uyumluluk) sağlanmalıdır:

| Terim | Tanım |
|---|---|
| **Backward compatibility** | Yeni kodun, eski kod tarafından yazılmış veriyi okuyabilmesi |
| **Forward compatibility** | Eski kodun, yeni kod tarafından yazılmış veriyi okuyabilmesi |

**Backward compatibility** genellikle sağlanması daha kolay olan yöndür. Çünkü yeni kodu yazan kişi, eski kodun veriyi nasıl yazdığını bilir ve bunu açıkça yönetebilir.

**Forward compatibility** ise daha zordur; çünkü eski kodun, yeni sürümün eklediği şeyleri görmezden gelmesini gerektirir.

Forward compatibility'nin önemli bir zorluğu şudur: Diyelim ki bir record schema'sına yeni bir alan eklediniz. Yeni kod bu yeni alanı içeren bir kayıt oluşturup veritabanına yazdı. Ardından hâlâ eski versiyonda çalışan bir uygulama (yeni alanı tanımayan) bu kaydı okudu, güncelledi ve geri yazdı. Bu durumda beklenen davranış, eski kodun yeni alanı anlayamasa bile onu bozmamasıdır. Ancak eğer kayıt, bilinmeyen alanları açıkça korumayan bir model nesnesine decode ediliyorsa, bu veri kaybolabilir.

<a id='2'></a>
## 2. Formats for Encoding Data (Veri Kodlama Formatları)

Programlar verilerle genellikle en az iki farklı gösterim biçiminde çalışır:

1. **In-memory representation (bellek içi gösterim):** Veri, nesneler, struct'lar, listeler, diziler, hash tablolar, ağaçlar vb. biçiminde tutulur. Bu yapılar CPU tarafından verimli erişim ve işlem için optimize edilmiştir; genellikle pointer'lar kullanılır.

2. **Byte sequence (bayt dizisi):** Veriyi bir dosyaya yazmak ya da ağ üzerinden göndermek istediğinizde, onu bağımsız bir bayt dizisi olarak (örneğin bir JSON belgesi) encode etmeniz gerekir. Bir pointer başka bir süreç için anlamsız olduğundan, bu bayt dizisi gösterimi genellikle bellekteki veri yapılarından oldukça farklı görünür.

### Encoding ve Decoding

Bu iki gösterim arasında bir çeviri yapılması gerekir:

- **Encoding** (aynı zamanda **serialization** ya da **marshaling** olarak da bilinir): Bellek içi gösterimden bayt dizisine dönüşüm.
- **Decoding** (aynı zamanda **parsing**, **deserialization** ya da **unmarshaling** olarak da bilinir): Bayt dizisinden bellek içi gösterime geri dönüşüm.

> **Terminoloji notu:** *Serialization* terimi ne yazık ki transaction'lar bağlamında da kullanılır ve burada tamamen farklı bir anlam taşır. Bu karışıklığı önlemek için bu bölümde **encoding** terimi tercih edilmektedir.

Bazı durumlarda encoding/decoding'e gerek yoktur. Örneğin bir veritabanı diskten yüklenen sıkıştırılmış veri üzerinde doğrudan çalışıyorsa. Ayrıca **zero-copy** veri formatları da vardır; bunlar hem çalışma zamanında hem de disk/ağ üzerinde açık bir dönüşüm adımı olmaksızın kullanılmak üzere tasarlanmıştır. **Cap'n Proto** ve **FlatBuffers** bu tür formatlara örnek gösterilebilir.

Ancak çoğu sistem bellek içi nesneler ile düz bayt dizileri arasında dönüşüm yapmak zorundadır. Bu çok yaygın bir problem olduğundan, aralarından seçim yapılabilecek pek çok kütüphane ve encoding formatı mevcuttur.

<a id='3'></a>
## 3. Language-Specific Formats (Dile Özgü Formatlar)

Pek çok programlama dili, bellek içi nesneleri bayt dizilerine encode etmek için yerleşik destek sunar. Örneğin:
- Java: `java.io.Serializable`
- Python: `pickle`
- Ruby: `Marshal`
- Java için üçüncü taraf: `Kryo`

Bu encoding kütüphaneleri pratiktir; minimum ek kod yazarak bellek içi nesneleri kaydedip geri yüklemenize olanak sağlar. Ancak ciddi sorunları da vardır:

### Sorunlar

**1. Dil bağımlılığı:** Encoding genellikle belirli bir programlama diline özgüdür. Bu veriyi başka bir dilde okumak zorlaşır. Bu formatı kullanırsanız kendinizi uzun süre mevcut programlama diline kilitlemiş olursunuz ve farklı diller kullanan başka kuruluşlarla sistemlerinizi entegre etmek güçleşir.

**2. Güvenlik açıkları:** Aynı nesne tiplerini geri yükleyebilmek için decoding sürecinin rastgele sınıfları örneklendirebilmesi (instantiate) gerekir. Bu durum güvenlik sorunlarının sık görülen bir kaynağıdır. Bir saldırgan uygulamanızın rastgele bir bayt dizisini decode etmesini sağlayabilirse, keyfi sınıfları örneklendirebilir ve bu da genellikle uzaktan rastgele kod çalıştırma gibi son derece tehlikeli sonuçlara yol açabilir.

**3. Versiyonlama eksikliği:** Bu kütüphanelerde veri versiyonlama çoğu zaman sonradan düşünülen bir şeydir. Hızlı ve kolay veri encoding için tasarlandıklarından, forward ve backward compatibility gibi can sıkıcı problemleri sık sık göz ardı ederler.

**4. Verimsizlik:** CPU zamanı ve encode edilmiş yapının boyutu açısından verimlilik de genellikle sonradan düşünülen bir konudur. Örneğin Java'nın yerleşik serialization'ı, kötü performansı ve şişirilmiş encoding'iyle ünlüdür.

Bu nedenlerle, dilinizin yerleşik encoding'ini çok geçici amaçlar dışında kullanmak genellikle kötü bir fikir olarak değerlendirilir.

<a id='4'></a>
## 4. JSON, XML ve Binary Variants

Pek çok programlama dili tarafından yazılıp okunabilen standartlaştırılmış encoding'lere geçildiğinde, **JSON** ve **XML** bariz adaylar olarak öne çıkar; her ikisi de yaygın biçimde tanınan ve desteklenen formatlardır. **CSV** de popüler bir dilden bağımsız format olmakla birlikte yalnızca iç içe geçme (nesting) özelliği bulunmayan tablo biçimindeki verileri destekler.

JSON, XML ve CSV metin tabanlı (textual) formatlardır; dolayısıyla bir ölçüde insan tarafından okunabilir, ancak sözdizimi (syntax) tartışmalı bir konudur. Yüzeysel sözdizimi sorunlarının ötesinde bu formatların çeşitli problemleri de vardır:

### Sorunlar

**1. XML'in aşırı karmaşıklığı:** XML çoğu zaman gereğinden fazla ayrıntılı (verbose) ve gereksiz yere karmaşık olmakla eleştirilir.

**2. Sayıların encoding'inde belirsizlik:** XML ve CSV'de bir sayıyı, tesadüfen rakamlardan oluşan bir string'den ayırt edemezsiniz (harici bir schema'ya başvurmadıkça). JSON string'leri ve sayıları birbirinden ayırt eder; ancak tam sayılar (integers) ile kayan noktalı sayıları (floating-point numbers) birbirinden ayırt etmez ve herhangi bir precision (hassasiyet) belirtmez. Bu durum büyük sayılarla uğraşırken sorun yaratır; IEEE 754 double-precision kayan noktalı sayıyla 2⁵³'ten büyük tam sayılar tam olarak temsil edilemez, dolayısıyla JavaScript gibi kayan noktalı sayı kullanan dillerde bu sayılar parse edilirken hata oluşur. Örneğin X (Twitter) her post için 64-bit sayı kullanan bir ID üretir; API'den dönen JSON bu post ID'lerini hem JSON sayısı hem de ondalık string olarak iki kez içerir; bu, JavaScript uygulamalarının sayıları yanlış parse etmesini önlemek için uygulanmış bir geçici çözümdür.

**3. Binary string desteği eksikliği:** JSON ve XML Unicode karakter string'leri için iyi destek sunar; ancak binary string'leri (character encoding olmaksızın bayt dizileri) desteklemez. Binary string'ler yararlı bir özellik olduğundan, insanlar bu sınırlamayı aşmak için binary veriyi Base64 kullanarak metin olarak encode ederler. Bu çalışır, ancak biraz kaba bir çözümdür ve veri boyutunu yaklaşık üçte bir oranında artırır.

**4. Schema dilleri karmaşık:** XML Schema ve JSON Schema güçlüdür ve bu nedenle öğrenmesi ve uygulaması oldukça karmaşıktır. Sayılar ve binary string'ler gibi verilerin doğru yorumlanması schema'daki bilgilere bağlı olduğundan, XML/JSON Schema kullanmayan uygulamaların uygun encoding/decoding mantığını koda gömmesi gerekebilir.

**5. CSV'de schema eksikliği:** CSV'de herhangi bir schema yoktur; bu nedenle her satır ve sütunun anlamını uygulamanın tanımlaması gerekir. Bir uygulama değişikliği yeni bir satır veya sütun eklerse, bu değişikliği manuel olarak yönetmeniz gerekir. CSV ayrıca oldukça belirsiz bir formattır (bir değer virgül veya satır sonu karakteri içeriyorsa ne olur?). Kaçış (escaping) kuralları resmi olarak belirtilmiş olsa da tüm parser'lar bunları doğru uygulamaz.

### Sonuç

Tüm bu eksikliklere karşın JSON, XML ve CSV pek çok amaç için yeterince iyidir. Özellikle veri değişim formatı (data interchange format) olarak, yani bir kuruluştan diğerine veri göndermek için yaygın olmaya devam edeceklerdir. Bu tür durumlarda, insanlar format üzerinde uzlaştığı sürece ne kadar şık ya da verimli olduğu çoğu zaman önem taşımaz. Farklı kuruluşların herhangi bir konuda uzlaşmasının güçlüğü, diğer tüm endişeleri geri planda bırakır.

<a id='5'></a>
## 5. JSON Schema

**JSON Schema**, sistemler arasında veri alışverişi yapıldığında ya da veri depolamaya yazıldığında veriyi modellemek için yaygın olarak benimsenmiş bir standarttır. JSON Schema'yı şu bağlamlarda görmek mümkündür:

- **Web services** içinde **OpenAPI** web servis spesifikasyonunun bir parçası olarak
- **Schema registries** içinde; Confluent'ın Schema Registry'si ve Red Hat'ın Apicurio Registry'si gibi
- **Databases** içinde; PostgreSQL'in `pg_jsonschema` validator eklentisi ve MongoDB'nin `$jsonSchema` validator sözdizimi gibi

### JSON Schema'nın Özellikleri

JSON Schema spesifikasyonu bir dizi özellik sunar. Schema'lar `string`, `number`, `integer`, `object`, `array`, `boolean` ve `null` gibi standart primitive type'ları içerir. Bunun yanı sıra JSON Schema, geliştiricilerin alanlara (fields) kısıtlamalar (constraints) bindirmesine olanak tanıyan ayrı bir validation spesifikasyonu sunar. Örneğin bir `port` alanı minimum 1, maksimum 65535 değerine sahip olabilir.

### Open ve Closed Content Models

JSON Schema'ların **open** ya da **closed** content model'e sahip olması mümkündür:

- **Open content model:** Schema'da tanımlanmamış herhangi bir alanın herhangi bir veri tipiyle var olmasına izin verir. JSON Schema'da `additionalProperties` değeri `true` olarak ayarlandığında (varsayılan budur) open content model etkinleşir. Dolayısıyla JSON Schema'lar genellikle neyin *izinli* olduğunu değil, neyin *izinsiz* olduğunu (yani tanımlı alanların herhangi birindeki geçersiz değerleri) tanımlar.

- **Closed content model:** Yalnızca açıkça tanımlanmış alanlara izin verir.

Open content model'ler güçlüdür ancak karmaşık olabilir. Örneğin tam sayılardan (ID'ler gibi) string'lere bir map tanımlamak istediğinizi düşünün. JSON'un tam sayı anahtarlarına izin veren bir map ya da dictionary tipi yoktur; JSON nesneleri her zaman string'leri anahtar olarak kullanır. İhtiyacınıza uyum sağlamak için bu tipi JSON Schema ile kısıtlayabilirsiniz; anahtarların yalnızca rakam içerebileceğini ve değerlerin yalnızca string olabileceğini `patternProperties` ve `additionalProperties` kullanarak belirtebilirsiniz.

```json
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "patternProperties": {
    "^[0-9]+$": {
      "type": "string"
    }
  },
  "additionalProperties": false
}
```

Open ve closed content model'lerin ve validator'ların yanı sıra JSON Schema; koşullu `if/else` schema mantığını, adlandırılmış tipleri, uzak schema'lara referansları ve çok daha fazlasını destekler. Tüm bu özellikler JSON Schema'yı son derece güçlü bir schema dili haline getirir. Ancak bu özellikler aynı zamanda kullanışsız tanımlamalara da yol açabilir. Uzak schema'ları çözümlemek, koşullu kurallar hakkında akıl yürütmek ya da schema'ları forward veya backward compatible biçimde geliştirmek zorlu olabilir. Benzer endişeler XML Schema için de geçerlidir.

<a id='6'></a>
## 6. Binary Encodings (İkili Kodlamalar)

JSON, XML'e kıyasla daha az ayrıntılıdır; ancak her ikisi de binary formatlara kıyasla çok fazla yer kaplar. Bu gözlem, JSON için (MessagePack, CBOR, BSON, BJSON, UBJSON, BISON, Hessian ve Smile gibi) ve XML için (WBXML ve Fast Infoset gibi) pek çok binary encoding'in geliştirilmesine yol açmıştır. Bu formatlar daha kompakt ve zaman zaman daha hızlı parse edilebilir olmalarından dolayı çeşitli nişlerde benimsenmişlerdir; ancak hiçbiri JSON ve XML'in metinsel sürümleri kadar yaygın kullanım görmemiştir.

Bu formatlardan bazıları veri tipi kümesini genişletir (örneğin tam sayılarla kayan noktalı sayıları birbirinden ayırt etmek ya da binary string desteği eklemek için), ancak JSON/XML veri modelini değiştirmeden korur. Özellikle bir schema öngörmediklerinden, encode edilmiş verinin içinde tüm nesne alan adlarını barındırmaları gerekir.

### MessagePack Örneği

Aşağıdaki JSON belgesini düşünelim:

```json
{
    "userName": "Martin",
    "favoriteNumber": 1337,
    "interests": ["daydreaming", "hacking"]
}
```

**MessagePack** ile encode edildiğinde 66 bayt üretilir. MessagePack bayt dizisinin ilk birkaç baytı şöyle yorumlanır:

- İlk bayt `0x83`: Arkasından 3 alanlı bir nesne (object) geldiğini belirtir (en anlamlı 4 bit `0x80` = nesne, en az anlamlı 4 bit `0x03` = 3 alan). Eğer bir nesnenin 15'ten fazla alanı varsa farklı bir type indicator kullanılır ve alan sayısı 2 ya da 4 bayta kodlanır.
- İkinci bayt `0xa8`: Arkasından 8 bayt uzunluğunda bir string geldiğini belirtir (en anlamlı 4 bit `0xa0` = string, en az anlamlı 4 bit `0x08` = 8 bayt uzunluk).
- Sonraki 8 bayt: ASCII'de `userName` alan adı. Uzunluk daha önce belirtildiğinden, string'in nerede bittiğini bildiren bir belirteç gerekmez.
- Sonraki 7 bayt: `0xa6` öneki ile 6 harflik `Martin` string değerini encode eder.

Binary encoding **66 bayt** uzunluğundadır; bu değer, whitespace kaldırılmış metinsel JSON encoding'in kapladığı 81 bayttan yalnızca biraz azdır. JSON'un tüm binary encoding'leri bu bakımdan birbirine benzer. Bu küçük alan tasarrufu (ve muhtemelen parsing hızındaki artış), insan tarafından okunabilirlik kaybına değer mi sorusu tartışmalıdır.

Aynı kaydı yarı sayıda bayta encode edebilen daha iyi yaklaşımlar mevcuttur: Protocol Buffers ve Avro.

In [1]:
# MessagePack encoding büyüklüğünü JSON ile karşılaştıran demo
import json

record = {
    "userName": "Martin",
    "favoriteNumber": 1337,
    "interests": ["daydreaming", "hacking"]
}

# JSON encoding boyutu (whitespace olmadan)
json_bytes = json.dumps(record, separators=(',', ':')).encode('utf-8')
print(f"JSON encoding boyutu: {len(json_bytes)} bayt")
print(f"JSON içeriği: {json_bytes.decode()}")

# MessagePack yaklaşık 66 bayt, Protocol Buffers 33 bayt, Avro 32 bayt üretir
print("\nKarşılaştırma:")
print("  JSON (whitespace'siz): ~81 bayt")
print("  MessagePack (binary JSON): ~66 bayt")
print("  Protocol Buffers: ~33 bayt")
print("  Avro: ~32 bayt")

JSON encoding boyutu: 81 bayt
JSON içeriği: {"userName":"Martin","favoriteNumber":1337,"interests":["daydreaming","hacking"]}

Karşılaştırma:
  JSON (whitespace'siz): ~81 bayt
  MessagePack (binary JSON): ~66 bayt
  Protocol Buffers: ~33 bayt
  Avro: ~32 bayt


<a id='7'></a>
## 7. Protocol Buffers

**Protocol Buffers** (protobuf), Google tarafından geliştirilen bir binary encoding kütüphanesidir. Facebook tarafından geliştirilen **Apache Thrift** ile benzerdir; bu bölümde Protocol Buffers hakkında söylenenler büyük ölçüde Thrift için de geçerlidir.

Protocol Buffers, encode edilecek her veri için bir **schema** gerektirir. Schema'yı **IDL (Interface Definition Language)** kullanarak şu şekilde tanımlarsınız:

```protobuf
syntax = "proto3";

message Person {
    string user_name       = 1;
    int64  favorite_number = 2;
    repeated string interests = 3;
}
```

Protocol Buffers, bu schema tanımını alarak çeşitli programlama dillerinde schema'yı uygulayan sınıflar (classes) üreten bir **code generation** (kod üretme) aracıyla birlikte gelir. Uygulama kodunuz, schema'ya uyan kayıtları encode veya decode etmek için bu üretilmiş kodu çağırabilir. Schema dili, JSON Schema'ya kıyasla çok daha basittir; yalnızca her kaydın alanlarını ve tiplerini tanımlar, alanlardaki olası değerlere ilişkin karmaşık validation kurallarını desteklemez.

### Protocol Buffers Binary Encoding'i Nasıl Çalışır?

Örnek JSON kaydını Protocol Buffers encoder ile encode etmek **33 bayt** üretir; bu MessagePack'in kapladığı 66 baytın yarısıdır.

MessagePack'ten farklı olarak, bu örnekte alan adları (`userName`, `favoriteNumber`, `interests`) yoktur. Bunların yerine encode edilmiş veri, schema tanımında yer alan sayılar olan **field tags** (alan etiketleri) içerir (`1`, `2`, `3`). Field tag'ler alan için birer alias işlevi görür; bu, hangi alanı kastettiğimizi alan adını yazmak zorunda kalmadan kompakt biçimde belirtmenin yoludur.

Protocol Buffers, alan tipini ve tag numarasını tek bir bayta sıkıştırarak ek alan tasarrufu sağlar. Ayrıca **variable-length integers** (değişken uzunluklu tam sayılar) kullanır: 1337 sayısı iki bayta encode edilir; her baytın en üstteki biti, daha fazla bayt olup olmadığını belirtmek için kullanılır. Bu yöntemle -64 ile 63 arasındaki sayılar tek bayta, -8192 ile 8191 arasındakiler iki bayta encode edilir. Büyük sayılar daha fazla bayt kullanır.

Protocol Buffers'ın açık bir liste ya da dizi veri tipi yoktur. Bunun yerine `interests` alanındaki `repeated` modifier, alanın tek bir değer yerine değerler listesi içerdiğini belirtir. Binary encoding'de liste elemanları, aynı kaydın içinde aynı field tag'in tekrarlanan oluşumları olarak gösterilir.

<a id='8'></a>
## 8. Field Tags ve Schema Evolution

Schema'ların zamanla değişmesi kaçınılmazdır; buna **schema evolution** denir. Protocol Buffers bu değişimi backward ve forward compatibility'yi koruyarak nasıl yönetir?

### Field Tag'lerin Önemi

Encode edilmiş bir kayıt, encode edilmiş alanların birleşiminden oluşur. Her alan, tag numarasıyla (örnek schema'daki `1`, `2`, `3` sayıları) tanımlanır ve bir veri tipiyle (string veya integer gibi) annotate edilir. Bir alanın değeri belirlenmemişse encode edilmiş kayıttan çıkarılır. Buradan şu sonuç çıkar: Field tag'ler, encode edilmiş verinin anlamı için kritik öneme sahiptir. Schema'daki bir alanın adını değiştirebilirsiniz; çünkü encode edilmiş veri hiçbir zaman alan adlarına başvurmaz. Ancak bir alanın tag'ini değiştiremezsiniz; aksi takdirde mevcut tüm encode edilmiş veri geçersiz hale gelir.

### Yeni Alan Ekleme

Schema'ya her birine yeni bir tag numarası verdiğiniz sürece yeni alanlar ekleyebilirsiniz:

- **Forward compatibility:** Eski kod, yeni kodun yazdığı veriyi okursa yeni tag numarasını tanımaz ve bu alanı yok sayar. Veri tipi annotation, parser'ın kaç baytı atlayacağını belirlemesine olanak tanır. Bu sayede Figure 5-1'deki veri kaybı sorunu önlenmiş olur.

- **Backward compatibility:** Her alan benzersiz bir tag numarasına sahip olduğu sürece yeni kod eski veriyi okuyabilir; çünkü tag numaraları aynı anlamı taşımaya devam eder. Yeni schema'ya bir alan eklenirse ve bu alanı içermeyen eski veri okunursa, alan varsayılan değeriyle doldurulur (string tipi için boş string, sayı için 0 gibi).

### Alan Silme

Alan silmek, alan eklemekle benzerdir; yalnızca backward ve forward compatibility endişeleri yer değiştirir. Aynı tag numarası tekrar kullanılamaz; çünkü eski tag numarasını içeren veriler hâlâ bir yerlerde bulunuyor olabilir ve bu alan yeni kod tarafından görmezden gelinmelidir. Geçmişte kullanılmış tag numaraları, unutulmamaları için schema tanımında **reserved** (ayrılmış) olarak işaretlenebilir.

### Veri Tipini Değiştirme

Bir alanın veri tipini değiştirmek bazı tipler için mümkündür; ancak değerlerin truncate edilmesi riski vardır. Örneğin 32-bit bir tam sayıyı 64-bit'e çevirdiğinizi varsayalım. Yeni kod, parser eksik bitleri 0 ile doldurabileceğinden eski kodun yazdığı veriyi kolayca okuyabilir. Ancak eski kod yeni kodun yazdığı veriyi okursa, 64-bit değeri tutmak için hâlâ 32-bit bir değişken kullanıyor olacaktır. Decode edilmiş 64-bit değer 32 bite sığmazsa truncate edilir.

<a id='9'></a>
## 9. Avro

**Apache Avro**, Protocol Buffers'dan bazı ilginç farklılıkları olan başka bir binary encoding formatıdır. Protocol Buffers'ın Hadoop'un kullanım senaryolarına uygun olmadığının görülmesinin ardından 2009 yılında Hadoop'un bir alt projesi olarak hayata geçirilmiştir.

Avro da encode edilecek verinin yapısını belirtmek için bir **schema** kullanır. İki schema dili sunar: İnsan tarafından düzenlemeye yönelik **Avro IDL** ve daha kolay makine tarafından okunabilen JSON tabanlı format. Protocol Buffers gibi, schema dilleri yalnızca alanları ve tiplerini belirtir; JSON Schema'daki gibi karmaşık validation kurallarını desteklemez.

```
record Person {
    string               userName;
    union { null, long } favoriteNumber = null;
    array<string>        interests;
}
```

Bu schema **tag numarası içermez**. Bu kaydı bu schema ile encode ettiğinizde Avro binary encoding yalnızca **32 bayt** üretir; bu, gördüğümüz tüm encoding'ler arasında en kompakt olanıdır.

### Avro Binary Encoding Nasıl Çalışır?

Bayt dizisini incelediğinizde alanları ya da veri tiplerini tanımlayan hiçbir şeyin olmadığını görürsünüz. Encoding, yalnızca birleştirilmiş değerlerden oluşur. Bir string, yalnızca bir uzunluk öneki ve ardından UTF-8 baytlarıdır; ancak encode edilmiş veride bunun string olduğunu söyleyen hiçbir şey yoktur; aynı zamanda bir integer ya da tamamen farklı bir şey de olabilir. Tam sayılar variable-length encoding ile encode edilir.

Binary veriyi parse etmek için schema'da göründükleri sırayla alanlar üzerinden geçer ve her alanın veri tipini belirlemek için schema'yı kullanırsınız. Bu, binary verinin yalnızca veriyi yazan kodun kullandığıyla **aynı schema**'yı kullanan kod tarafından doğru decode edilebildiği anlamına gelir. Reader ile writer arasındaki herhangi bir schema uyumsuzluğu yanlış decode edilmiş veriye yol açar.

### Writer's Schema ve Reader's Schema

Peki Avro schema evolution'ı nasıl destekler?

Bir uygulama bazı verileri encode etmek istediğinde (bir dosyaya veya veritabanına yazmak, ağ üzerinden göndermek vb. için) uygulamanın bildiği schema sürümünü kullanır; bu **writer's schema** olarak bilinir.

Bazı verileri decode etmek için bir uygulama iki schema kullanır:
- **Writer's schema:** Encoding için kullanılanın aynısı
- **Reader's schema:** Farklı olabilir; uygulama kodunun beklediği alanları ve tiplerini tanımlar

Reader'ın ve writer'ın schema'ları aynıysa, decode işlemi kolaydır. Farklıysa Avro, ikisini karşılaştırarak veriyi writer'ın schema'sından reader'ın schema'sına çevirerek farkları çözer.

Avro spesifikasyonu bu çözümlemenin nasıl gerçekleştiğini tam olarak tanımlar:
- Writer'ın schema'sı ile reader'ın schema'sının alanlarının farklı sırada olması sorun değildir; çünkü schema çözümlemesi alanları **alan adına göre** eşleştirir.
- Kod, writer'ın schema'sında görünen ancak reader'ın schema'sında olmayan bir alanla karşılaşırsa onu yok sayar.
- Kod belirli bir alan bekliyorsa ancak writer'ın schema'sında bu adda bir alan yoksa reader'ın schema'sındaki varsayılan değerle doldurulur.

### Schema Evolution Kuralları

Avro'da:
- **Forward compatibility:** Writer, reader'dan daha yeni bir schema sürümü kullanabilir.
- **Backward compatibility:** Writer, reader'dan daha eski bir schema sürümü kullanabilir.

Uyumluluğu korumak için yalnızca **varsayılan değeri olan** bir alanı ekleyebilir ya da kaldırabilirsiniz. Örneğin varsayılan değeri olan yeni bir alan eklediğinizde bu yeni alan yeni schema'da bulunurken eskisinde yoktur. Yeni schema kullanan bir reader, eski schema ile yazılmış bir kaydı okuduğunda eksik alan için varsayılan değer kullanılır.

Varsayılan değeri olmayan bir alan eklerseniz, yeni reader'lar eski writer'ların yazdığı veriyi okuyamaz; bu backward compatibility'yi bozar. Varsayılan değeri olmayan bir alanı kaldırırsanız, eski reader'lar yeni writer'ların yazdığı veriyi okuyamaz; bu forward compatibility'yi bozar.

Bazı programlama dillerinde `null` herhangi bir değişken için kabul edilebilir bir varsayılan değerdir, ancak Avro'da bu böyle değildir. Bir alanın `null` olmasına izin vermek istiyorsanız **union type** kullanmanız gerekir. Örneğin `union { null, long, string } field;` ifadesi `field`'ın sayı, string ya da `null` olabileceğini belirtir. `null`'ı varsayılan değer olarak yalnızca union'ın ilk dalıysa kullanabilirsiniz. Bu, her şeyin varsayılan olarak nullable olmasından biraz daha ayrıntılıdır; ancak neyin `null` olup olamayacağı konusunda açık olmak, hataları önlemeye yardımcı olur.

### Writer's Schema'yı Belirleme

Reader, belirli bir veri parçasını encode etmek için hangi schema'nın kullanıldığını nasıl bilebilir? Her kayıtla birlikte schema'nın tamamını ekleyemeyiz; çünkü schema büyük olasılıkla encode edilmiş veriden çok daha büyük olacak ve binary encoding'den elde edilen tüm alan tasarrufunu ortadan kaldıracaktır.

Yanıt, Avro'nun kullanım bağlamına göre değişir:

- **Large file with lots of records (kayıtlarla dolu büyük dosya):** Avro'nun yaygın kullanım alanlarından biri, milyonlarca kaydı içeren ve tümü aynı schema ile encode edilmiş büyük dosyaları depolamaktır. Bu durumda dosyanın writer'ı schema'yı dosyanın başına yalnızca bir kez ekleyebilir. Avro bunun için **object container files** adlı bir dosya formatı belirtir.

- **Database with individually written records (tek tek yazılmış kayıtlara sahip veritabanı):** Bir veritabanında farklı kayıtlar farklı zamanlarda farklı schema'larla yazılmış olabilir. Bu durumda en basit çözüm, her encode edilmiş kaydın başına bir sürüm numarası eklemek ve bir schema sürümleri listesini veritabanında tutmaktır. Reader bir kaydı alabilir, sürüm numarasını çıkarabilir ve ardından veritabanından o sürüm numarasına karşılık gelen writer'ın schema'sını alabilir. Confluent'ın Apache Kafka için schema registry'si ve LinkedIn'in Espresso'su bu şekilde çalışır.

- **Sending records over a network connection (ağ bağlantısı üzerinden kayıt gönderme):** İki süreç çift yönlü bir ağ bağlantısı üzerinden iletişim kurduğunda, bağlantı kurulumunda schema sürümünü müzakere edebilirler ve ardından bu schema'yı bağlantının ömrü boyunca kullanabilirler. Avro RPC protokolü bu şekilde çalışır.

Schema sürümlerinin veritabanı her durumda faydalıdır; belgeleme işlevi görür ve schema değişikliğinin uyumluluğunu kontrol etme fırsatı verir.

### Dynamically Generated Schemas

Avro'nun yaklaşımının Protocol Buffers'a kıyasla bir avantajı, schema'nın tag numarası içermemesidir. Bu neden önemlidir?

Avro, **dynamically generated schemas** (dinamik olarak üretilen schema'lar) için daha elverişlidir. Örneğin içeriğini bir dosyaya dökmek istediğiniz bir ilişkisel veritabanınız olduğunu düşünün. Avro kullanırsanız, ilişkisel schema'dan oldukça kolay bir şekilde bir Avro schema'sı oluşturabilirsiniz; bu schema her tablonun bir kayıt schema'sı oluşturuyor ve her sütunun o kayıtta bir alan haline geldiği bir yapıdır. Veritabanındaki sütun adı Avro'daki alan adına eşlenir.

Veritabanı schema'sı değişirse yeni bir Avro schema'sı oluşturulabilir ve veri yeni schema ile export edilebilir. Bu süreç schema değişikliğine herhangi bir dikkat göstermeden çalışabilir. Yeni veri dosyalarını okuyan herkes kaydın alanlarının değiştiğini görecektir; ancak alanlar ada göre tanımlandığından, güncellenmiş writer schema'sı eski reader schema'sıyla eşleştirilebilir.

Oysa Protocol Buffers kullanırsaydınız, field tag'lerin büyük olasılıkla elle atanması gerekirdi. Veritabanı schema'sı her değiştiğinde, bir yöneticinin veritabanı sütun adlarından field tag'lere olan eşlemeyi manuel olarak güncellemesi gerekirdi. Bu tür dinamik olarak üretilen schema, Protocol Buffers'ın tasarım hedefleri arasında yer almazken Avro'nun hedefleri arasındaydı.

<a id='10'></a>
## 10. The Merits of Schemas (Schema'ların Faydaları)

Protocol Buffers ve Avro'nun her ikisi de bir binary encoding formatını tanımlamak için schema kullandığını gördük. Schema dilleri, daha ayrıntılı validation kurallarını (örneğin "bu alanın string değeri bu regular expression ile eşleşmelidir" ya da "bu alanın integer değeri 0 ile 100 arasında olmalıdır") destekleyen XML Schema veya JSON Schema'ya kıyasla çok daha basittir. Protocol Buffers ve Avro uygulaması ve kullanımı daha kolay olduğundan, oldukça geniş bir programlama dili yelpazesinde destek görmüştür.

Bu encoding'lerin dayandığı fikirler hiç de yeni değildir. Örneğin 1984'te ilk kez standartlaştırılan bir schema tanımlama dili olan **ASN.1** ile pek çok ortak noktaları vardır. ASN.1, çeşitli ağ protokollerini tanımlamak için kullanılmıştır ve binary encoding'i (DER) hâlâ SSL sertifikalarını (X.509) encode etmek için kullanılmaktadır. ASN.1, Protocol Buffers'a benzer şekilde tag numaraları kullanarak schema evolution'ı destekler. Ancak son derece karmaşıktır ve belgeleri yetersizdir; dolayısıyla yeni uygulamalar için iyi bir seçenek değildir.

Birçok veri sistemi de verileri için özel binary encoding uygulamaları içerir. Örneğin çoğu ilişkisel veritabanı, sorguları veritabanına göndermek ve yanıtları geri almak için üzerinden iletişim kurulabilen bir ağ protokolüne sahiptir. Bu protokoller genellikle belirli bir veritabanına özgüdür ve veritabanı satıcısı, veritabanının ağ protokolünden gelen yanıtları bellek içi veri yapılarına decode eden bir driver (ODBC veya JDBC API'leri aracılığıyla) sağlar.

### Schema Tabanlı Binary Encoding'lerin Avantajları

Metin tabanlı veri formatlarının (JSON, XML ve CSV) yaygın olmasına karşın, schema tabanlı binary encoding'ler de geçerli bir seçenektir. Çeşitli güzel özelliklere sahiptirler:

1. **Kompaktlık:** Alan adlarını encode edilmiş veriden çıkarabildiklerinden, çeşitli "binary JSON" varyantlarından çok daha kompakt olabilirler.

2. **Belgeleme:** Schema, değerli bir belgeleme biçimidir. Decode için schema gerektiğinden, bu belgelemenin güncel olduğundan emin olabilirsiniz (manuel olarak tutulan belgeler gerçeklikten kolayca sapabilir).

3. **Uyumluluk kontrolü:** Schema veritabanı tutmak, herhangi bir şey deploy edilmeden önce schema değişikliklerinin forward ve backward compatibility'sini kontrol etmenize olanak tanır.

4. **Code generation:** Statik tipli (statically typed) programlama dilleri kullanıcıları için schema'dan kod üretme yeteneği faydalıdır; compile time'da tip kontrolü yapılmasına olanak tanır.

Özetle, schema evolution; schemaless/schema-on-read JSON veritabanlarının sağladığıyla aynı esneklik türüne olanak tanırken aynı zamanda veriniz hakkında daha iyi garantiler ve daha iyi tooling sunar. Yine de operasyonları basit tutmak için eş zamanlı schema format sayısını en aza indirmek tavsiye edilir.

<a id='11'></a>
## 11. Modes of Dataflow (Veri Akışı Modları)

Bölümün başında şunu belirtmiştik: Belleği paylaşmadığınız başka bir süreçle veri göndermek istediğinizde, örneğin veriyi ağ üzerinden göndermek ya da dosyaya yazmak istediğinizde, bunu bir bayt dizisi olarak encode etmeniz gerekir.

**Forward ve backward compatibility** önemlidir; bunlar evolvability için kritiktir (sisteminizin parçalarını her şeyi aynı anda değiştirmek zorunda kalmadan bağımsız olarak güncellemenize izin vererek değişimi kolaylaştırır). Compatibility, veriyi encode eden süreçle onu decode eden süreç arasındaki ilişkidir.

Bu oldukça soyut bir fikirdir; çünkü veriler bir süreçten diğerine pek çok farklı yolla akabilir. Kodu kim encode eder, kodu kim decode eder?

Verinin süreçler arasında akmasının en yaygın yolları:

- **Databases (Veritabanları)** aracılığıyla
- **Service calls (Servis çağrıları)** aracılığıyla
- **Workflow engines** aracılığıyla
- **Asynchronous messages (Asenkron mesajlar)** aracılığıyla

<a id='12'></a>
## 12. Dataflow Through Databases

Bir veritabanında, yazan süreç veriyi encode eder ve okuyan süreç onu decode eder. Veritabanına yalnızca tek bir süreç erişiyor olabilir; bu durumda reader, aynı sürecin sonraki bir sürümüdür. Bu senaryoda veritabanına bir şey depolamayı **geleceğinizdeki kendinize mesaj göndermek** olarak düşünebilirsiniz. Backward compatibility burada açıkça gereklidir; aksi takdirde gelecekteki kendiniz daha önce yazdıklarınızı decode edemez.

Genel olarak birden fazla sürecin aynı anda bir veritabanına erişmesi yaygın bir durumdur. Bu süreçler farklı uygulamalar veya servisler olabilir ya da basitçe aynı servisin birden fazla instance'ı olabilir (ölçeklenebilirlik veya hata toleransı için paralel olarak çalışıyor). Her iki durumda da böyle bir ortamda veritabanına erişen bazı süreçlerin daha yeni kodu, bazılarının ise daha eski kodu çalıştırması olasıdır. Bu nedenle veritabanlarında **forward compatibility** de sık sık gereklidir.

### Data Outlives Code (Veri Kodun Ömrünü Geçer)

Bir veritabanı genellikle herhangi bir değerin herhangi bir zamanda güncellenmesine izin verir. Tek bir veritabanı içinde beş milisaniye önce yazılmış değerler ile beş yıl önce yazılmış değerler bir arada bulunabilir.

Uygulamanızın yeni bir sürümünü deploy ettiğinizde, eski sürümü yeni sürümle birkaç dakika içinde tamamen değiştirmiş olabilirsiniz. Veritabanı içerikleri için bu böyle değildir; o tarihten bu yana açıkça yeniden yazmadıysanız, beş yıllık veri hâlâ orijinal encoding'iyle orada duruyor olacaktır. Bu gözlem zaman zaman **"data outlives code"** (veri kodun ömrünü geçer) olarak özetlenir.

Veriyi yeni bir schema'ya yeniden yazmak (**migrating**) kesinlikle mümkündür, ancak büyük bir veri kümesinde bu işlem pahalıdır. Bu nedenle çoğu veritabanı bu işlemi erteler ve asenkron ile en iyi çaba esasına dayalı olarak gerçekleştirir. Örneğin:

- LSM-tree storage engine'ler veriyi compaction sırasında en son formatla yeniden yazar.
- Çoğu relational database, mevcut veriyi yeniden yazmaksızın `null` varsayılan değeriyle yeni bir sütun ekleme gibi basit schema değişikliklerine izin verir. Eski bir satır okunduğunda, veritabanı diskteki encode edilmiş veriden eksik olan sütunlar için `null` değerini doldurur.

Schema evolution böylece tüm veritabanının tek bir schema ile encode edilmiş gibi görünmesini sağlar; oysa altta yatan depolama alanı schema'nın çeşitli geçmiş sürümleriyle encode edilmiş kayıtlar içeriyor olabilir.

Tek değerli bir attribute'u çok değerli yapma veya bazı verileri ayrı bir tabloya taşıma gibi daha karmaşık schema değişiklikleri, genellikle uygulama düzeyinde veri yeniden yazımını gerektirir.

### Archival Storage

Veritabanınızın zaman zaman anlık görüntüsünü (snapshot) aldığınızı düşünün; yedekleme ya da bir data warehouse'a yükleme amacıyla. Bu durumda data dump, kaynak veritabanındaki orijinal encoding farklı dönemlerden schema sürümleri içeriyor olsa bile genellikle en son schema kullanılarak encode edilir. Veriyi zaten kopyaladığınızdan, kopyanın tutarlı biçimde encode edilmesini de sağlayabilirsiniz.

Data dump tek seferde yazılıp ardından değiştirilemez olduğundan, Avro object container files gibi formatlar bu kullanım için iyi bir seçimdir. Ayrıca veriyi Parquet gibi analiz dostu bir column-oriented formatta encode etmek için iyi bir fırsattır.

<a id='13'></a>
## 13. Dataflow Through Services: REST and RPC

Süreçlerin ağ üzerinden iletişim kurması gerektiğinde, iletişimi birkaç farklı biçimde düzenleyebilirsiniz. En yaygın düzenleme iki rol barındırır: **clients** (istemciler) ve **servers** (sunucular). Server'lar ağ üzerinde bir **API** sunar ve client'lar bu API'ye istek göndermek için server'lara bağlanabilir. Server'ın sunduğu API **service** (servis) olarak bilinir.

Web bu şekilde çalışır: Client'lar (web tarayıcıları) web server'lara istek gönderir; HTML, CSS, JavaScript, resimler vb. indirmek için `GET` isteği, sunucuya veri göndermek için `POST` isteği gönderir.

Servisler bir bakıma veritabanlarına benzer: Genellikle client'ların veri göndermesine ve sorgulamasına izin verirler. Ancak veritabanları keyfi sorgulara izin verirken, servisler servisin iş mantığı (uygulama kodu) tarafından önceden belirlenmiş girdi ve çıktılara izin veren uygulamaya özgü bir API sunar. Bu kısıtlama bir kapsülleme (encapsulation) derecesi sağlar: Servisler, client'ların ne yapıp yapamayacağına dair ayrıntılı kısıtlamalar uygulayabilir.

**Service-oriented/microservices architecture**'ın temel tasarım hedefi, servisleri bağımsız olarak deploy edilebilir ve geliştirilebilir kılarak uygulamayı değiştirmeyi ve bakımını yapmayı kolaylaştırmaktır.

### Web Services

HTTP, servisle konuşmak için kullanılan protokol olduğunda, buna **web service** denir. Web servisler şu bağlamlarda yaygın olarak kullanılır:

1. Kullanıcının cihazında çalışan bir client uygulaması (mobil cihazdaki yerel uygulama veya tarayıcıdaki JavaScript web uygulaması) HTTP üzerinden bir servise istek gönderiyor.
2. Bir servis, servis odaklı/microservices mimarisinin bir parçası olarak genellikle aynı özel ağda bulunan başka bir servise istek gönderiyor.
3. Bir servis, başka bir kuruluşun sahip olduğu bir servise istek gönderiyor; bu genellikle internet üzerinden gerçekleşir.

En popüler servis tasarım felsefesi, HTTP'nin prensipleri üzerine inşa edilen **REST (Representational State Transfer)**'dir. REST; basit veri formatlarını, kaynakları tanımlamak için URL'leri ve önbellek kontrolü, kimlik doğrulama ve içerik türü müzakeresi için HTTP özelliklerini kullanmayı vurgular. REST prensipleri doğrultusunda tasarlanmış bir API'ye **RESTful** denir.

Servis geliştiriciler sıklıkla servislerinin API endpoint'lerini ve veri modellerini tanımlamak ve zamanla geliştirmek için bir **IDL (Interface Definition Language)** kullanır. En popüler iki servis IDL'si şunlardır:

- **OpenAPI** (Swagger olarak da bilinir): JSON gönderip alan web servisler için
- **Protocol Buffers IDL**: gRPC servisleri için

**Service framework**'ler (Spring Boot, FastAPI, gRPC gibi) bu çabayı basitleştirmek için benimsenir. Service framework'ler geliştiricilerin her API endpoint'in iş mantığını yazmaya odaklanmasına olanak tanırken framework kodu routing, metrics, caching ve authentication gibi işlemleri yönetir.

### Remote Procedure Calls (RPC) ve Sorunları

Web servisleri, 1970'lerden beri var olan **RPC (Remote Procedure Call)** teknolojileri serisinin en son halkasıdır. Enterprise JavaBeans (EJB) ve Java'nın Remote Method Invocation (RMI) yalnızca Java ile sınırlıdır. Distributed Component Object Model (DCOM) Microsoft platformlarıyla sınırlıdır. Common Object Request Broker Architecture (CORBA) aşırı karmaşıktır ve backward ya da forward compatibility sağlamaz. SOAP ve WS-* web services framework, farklı satıcılar arasında birlikte çalışabilirlik sağlamayı amaçlar ancak karmaşıklık ve uyumluluk sorunlarıyla boğuşur.

**RPC modeli**, uzak bir ağ servisine yapılan isteğin, aynı süreç içinde bir fonksiyon veya metot çağrısıyla aynı görünmesini sağlamaya çalışır. Bu soyutlamaya **location transparency** (konum şeffaflığı) denir. Kulağa pratik gelse de bu yaklaşım temelden hatalıdır; çünkü ağ isteği yerel bir fonksiyon çağrısından temelden farklıdır:

1. **Tahmin edilemezlik:** Yerel fonksiyon çağrısı kontrol edebileceğiniz parametrelere bağlıdır. Ağ isteği ise tamamen kontrolünüz dışındaki nedenlerle (ağ sorunu, uzak makine yavaş veya erişilemez olabilir) tahmin edilemezdir.

2. **Timeout:** Yerel fonksiyon çağrısı ya bir sonuç döndürür, ya istisna (exception) fırlatır, ya da hiç dönmez. Ağ isteğinin ek bir sonuç olasılığı vardır: **timeout** nedeniyle sonuçsuz dönebilir. Bu durumda ne olduğunu bilemezsiniz.

3. **Idempotence sorunu:** Başarısız ağ isteğini yeniden denerseniz, önceki isteğin ulaşmış olup yalnızca yanıtın kaybolduğu olabilir. Bu durumda yeniden deneme, eylemi birden fazla kez gerçekleştirecektir; protokole **idempotence** (tekrarsallık) için bir deduplication mekanizması yerleştirmedikçe.

4. **Değişken gecikme (latency):** Yerel fonksiyon çağrısı genellikle her seferinde aynı süreyi alır. Ağ isteği fonksiyon çağrısından çok daha yavaştır; iyi zamanlarda bir milisaniyenin altında tamamlanabilirken ağ sıkışıklığı veya uzak servis aşırı yüklendiğinde aynı işlemi yapmak saniyeler alabilir.

5. **Parametre aktarımı:** Yerel fonksiyon çağrısında yerel belleğe referanslar (pointers) verimli biçimde aktarabilirsiniz. Ağ üzerinden ise tüm bu parametrelerin ağ üzerinden gönderilebilecek bir bayt dizisine encode edilmesi gerekir.

6. **Dil uyumsuzluğu:** Client ve servis farklı programlama dillerinde uygulanmış olabilir; bu nedenle RPC framework'ün veri tiplerini bir dilden diğerine çevirmesi gerekir.

Tüm bu faktörler, uzak servisi programlama dilinizde yerel bir nesne gibi görünmeye çalışmanın anlamsız olduğunu gösterir; çünkü bu temelden farklı bir şeydir.

### Data Encoding ve RPC için Evolvability

RPC client'larının ve server'larının bağımsız olarak değiştirilebilmesi ve deploy edilebilmesi için evolvability önemlidir. Veri akışı için basitleştirici bir varsayım yapabiliriz: Tüm server'ların önce, tüm client'ların sonra güncelleneceğini varsaymak makuldür. Dolayısıyla isteklerde yalnızca **backward compatibility**, yanıtlarda ise **forward compatibility** gereklidir.

API versiyonlamasının nasıl çalışması gerektiğine dair bir uzlaşı yoktur. RESTful API'ler için yaygın yaklaşımlar URL'de veya HTTP `Accept` header'ında bir sürüm numarası kullanmaktır.

<a id='14'></a>
## 14. Durable Execution and Workflows

Tanım gereği, servis tabanlı mimarilerin uygulamanın farklı bölümlerinden sorumlu birden fazla servisi vardır. Örneğin bir kredi kartını şarj edip fonları bir banka hesabına yatıran bir ödeme işleme uygulamasını düşünün. Bu sistemde büyük olasılıkla dolandırıcılık tespiti, kredi kartı entegrasyonu, banka entegrasyonu vb. için farklı servisler olacaktır.

Tek bir ödemenin işlenmesi çok sayıda servis çağrısı gerektirir. Bir ödeme işleyici servisi, dolandırıcılık tespiti için fraud detection servisini, kredi kartını borçlandırmak için credit card servisini, borçlandırılan fonları yatırmak için banking servisini çağırabilir. Bu adım dizisine **workflow** (iş akışı), her adıma ise **task** (görev) denir. Workflow'lar genellikle bir task grafiği olarak tanımlanır.

Workflow tanımları genel amaçlı bir programlama diliyle, domain-specific language (DSL) ile veya **Business Process Execution Language (BPEL)** gibi bir markup diliyle yazılabilir.

> **Not:** Farklı workflow engine'ler task'lar için farklı isimler kullanır. Temporal örneğin **activity** terimini kullanır. Diğerleri task'lara **durable functions** der. Adlar farklı olsa da kavramlar aynıdır.

### Workflow Engine

Workflow'lar bir **workflow engine** (iş akışı motoru) tarafından çalıştırılır veya yürütülür. Workflow engine'ler her task'ın ne zaman ve hangi makinede çalıştırılacağını, bir task başarısız olursa ne yapılacağını, kaç task'ın paralel olarak yürütülmesine izin verildiğini ve daha fazlasını belirler.

Workflow engine'ler tipik olarak bir **orchestrator** ve bir **executor**'dan oluşur:
- **Orchestrator:** Yürütülecek task'ları zamanlamaktan sorumludur.
- **Executor:** Task'ları yürütmekten sorumludur.

Çeşitli kullanım senaryolarını ele alan pek çok workflow engine türü mevcuttur:
- **Airflow, Dagster, Prefect:** Veri sistemleriyle entegre olur, ETL task'larını orkestre eder.
- **Camunda, Orkes:** Mühendis olmayanların daha kolay workflow tanımlayıp yürütebilmesi için BPMN gibi grafiksel notasyon sunar.
- **Temporal, Restate:** **Durable execution** sağlar.

### Durable Execution

**Durable execution** framework'leri, transactionality gerektiren servis tabanlı mimarileri inşa etmenin popüler bir yolu haline gelmiştir. Ödeme örneğimizde her ödemeyi tam olarak bir kez işlemek isteriz. Workflow yürütülürken bir hata, kredi kartı borçlandırması yapılmış ancak buna karşılık gelen banka hesabı yatırması gerçekleştirilmemiş bir duruma yol açabilir. Servis tabanlı bir mimaride iki task'ı bir veritabanı transaction'ına saramayız.

Durable execution framework'leri, workflow'lar için **exactly-once semantics** (tam olarak bir kez semantiği) sağlamanın bir yoludur. Bir task başarısız olursa framework task'ı yeniden yürütür; ancak başarısız olmadan önce task'ın başarıyla yaptığı tüm RPC çağrılarını veya state değişikliklerini atlayacaktır. Sanki bu çağrıyı yapmış gibi davranacak, ancak bunun yerine önceki çağrıdan sonuçları geri döndürecektir. Bu mümkündür; çünkü durable execution framework'leri tüm RPC'leri ve state değişikliklerini **write-ahead log** gibi dayanıklı bir depolamaya kaydeder.

Aşağıda Temporal kullanarak durable execution'ı destekleyen bir workflow tanımı örneği verilmiştir:

```python
@workflow.defn
class PaymentWorkflow:
    @workflow.run
    async def run(self, payment: PaymentRequest) -> PaymentResult:
        is_fraud = await workflow.execute_activity(
            check_fraud,
            payment,
            start_to_close_timeout=timedelta(seconds=15),
        )
        if is_fraud:
            return PaymentResultFraudulent
        credit_card_response = await workflow.execute_activity(
            debit_credit_card,
            payment,
            start_to_close_timeout=timedelta(seconds=15),
        )
        # ...
```

### Durable Execution'ın Zorlukları

Temporal gibi framework'lerin zorlukları da mevcuttur:

1. **Idempotency:** Üçüncü taraf ödeme gateway'i gibi dış servisler hâlâ idempotent API sağlamalıdır. Geliştiriciler yinelenen yürütmeyi önlemek için bu API'ler için benzersiz ID'ler kullanmayı unutmamalıdır.

2. **Kod değişikliği kırılganlığı:** Durable execution framework'leri her RPC çağrısını sırayla logladığından, sonraki yürütmelerin aynı RPC çağrılarını aynı sırayla yapmasını bekler. Bu, kod değişikliklerini kırılgan hale getirir; fonksiyon çağrılarının sırasını değiştirmeniz bile tanımsız davranışa yol açabilir. Mevcut bir workflow'un kodunu değiştirmek yerine kodu ayrıca yeni bir sürüm olarak deploy etmek daha güvenlidir; böylece mevcut workflow invocation'larının yeniden yürütmeleri eski sürümü kullanmaya devam eder.

3. **Determinizm gerekliliği:** Durable execution framework'leri tüm kodun deterministic (belirleyici) biçimde replay edilmesini bekler (aynı girdiler aynı çıktıları üretir). Bu nedenle random number generator veya system clock çağırma gibi nondeterministic kodlar sorunludur. Framework'ler genellikle bu tür kütüphane fonksiyonlarının kendi deterministic uygulamalarını sağlar; ancak bunları kullanmayı unutmamak gerekir.

> Kodu deterministic yapmak güçlü bir fikirdir ancak sağlam biçimde uygulamak zordur.

<a id='15'></a>
## 15. Event-Driven Architectures

Bu son bölümde, encode edilmiş verinin bir süreçten diğerine akmasının başka bir yolu olan **event-driven architectures** (olay güdümlü mimariler) üzerinde duracağız. Bu bağlamda bir isteğe **event** (olay) veya **message** (mesaj) denir.

RPC'den farklı olarak gönderen, alıcının olayı işlemesini genellikle beklemez. Ayrıca event'ler genellikle alıcıya doğrudan ağ bağlantısı üzerinden gönderilmez; bunun yerine **message broker** (mesaj aracısı) adı verilen, mesajı geçici olarak depolayan bir aracı üzerinden iletilir. Message broker'ın diğer isimleri: **event broker**, **message queue** veya **message-oriented middleware**.

### Message Broker Kullanmanın Avantajları

Doğrudan RPC ile karşılaştırıldığında message broker kullanmanın çeşitli avantajları vardır:

1. Alıcı kullanılamaz veya aşırı yüklenmiş durumdaysa buffer görevi görerek sistem güvenilirliğini artırır.
2. Çökmüş bir sürece mesajları otomatik olarak yeniden gönderebilir ve mesajların kaybolmasını önler.
3. Gönderenin alıcının IP adresine doğrudan bağlanmasına gerek olmadığından, service discovery ihtiyacını ortadan kaldırır.
4. Aynı mesajın birden fazla alıcıya gönderilmesine olanak tanır.
5. Göndereni alıcıdan mantıksal olarak ayrıştırır (gönderen yalnızca mesajları yayınlar ve kimin aldığıyla ilgilenmez).

Message broker aracılığıyla iletişim **asynchronous** (asenkron) bir niteliktedir: Gönderen mesajın iletilmesini beklemez; mesajı gönderir ve işi biter. Ancak gönderenin ayrı bir channel'da yanıt beklemesiyle synchronous RPC benzeri bir model uygulamak da mümkündür.

### Message Brokers

Geçmişte message broker ortamı TIBCO, IBM WebSphere ve webMethods gibi şirketlerin ticari kurumsal yazılımları tarafından domine edilirdi; daha sonra RabbitMQ, ActiveMQ, HornetQ, NATS, Redpanda ve Apache Kafka gibi açık kaynaklı uygulamalar popüler hale geldi. Daha yakın zamanda ise Amazon Kinesis, Azure Service Bus ve Google Cloud Pub/Sub gibi cloud servisler yaygınlaştı.

İki mesaj dağıtım deseni en sık kullanılır:

1. **Queue (Kuyruk):** Bir süreç adlandırılmış bir kuyruğa mesaj ekler ve kuyruğun **consumer**'ı (tüketicisi) mesajı alır. Birden fazla consumer varsa, bunlardan yalnızca biri mesajı alır.

2. **Topic (Konu):** Bir süreç adlandırılmış bir topic'e mesaj yayınlar (**publish**) ve broker bu mesajı söz konusu topic'in tüm **subscriber**'larına (abonelerine) iletir. Birden fazla subscriber varsa, hepsi mesajı alır.

Message broker'lar genellikle belirli bir veri modelini zorunlu kılmaz. Mesaj, bazı meta verilerle birlikte yalnızca bir bayt dizisidir; dolayısıyla herhangi bir encoding formatı kullanabilirsiniz. Yaygın bir yaklaşım Protocol Buffers, Avro veya JSON kullanmak ve tüm geçerli schema sürümlerini depolamak ve uyumluluklarını kontrol etmek için message broker'ın yanına bir **schema registry** deploy etmektir.

Message broker'lar mesajlarının dayanıklılığı (durability) açısından farklılık gösterir. Pek çok tanesi, message broker çökerse veya yeniden başlatılması gerekirse mesajların kaybolmaması için mesajları diske yazar. Veritabanlarından farklı olarak pek çok message broker, mesajlar tüketildikten sonra bunları otomatik olarak siler. Ancak bazı broker'lar mesajları süresiz depolamak üzere yapılandırılabilir; bu, **event sourcing** kullanmak istiyorsanız ihtiyaç duyacağınız bir özelliktir.

Bir consumer mesajları başka bir topic'e yeniden yayınlarsa (republish), daha önce veritabanları bağlamında açıklanan sorunu önlemek için bilinmeyen alanları korumaya dikkat etmeniz gerekebilir.

### Distributed Actor Frameworks

**Actor model**, tek bir süreçte concurrency (eş zamanlılık) için bir programlama modelidir. Thread'lerle (ve buna eşlik eden race condition, locking ve deadlock sorunlarıyla) doğrudan ilgilenmek yerine, mantık **actor**'lara kapsüllenir. Her actor genellikle tek bir client veya entity'yi temsil eder. Bazı yerel state'e sahip olabilir (başka herhangi bir actor ile paylaşılmaz) ve diğer actor'larla asenkron mesajlar göndererek ve alarak iletişim kurar. Mesaj iletimi garanti değildir; belirli hata senaryolarında mesajlar kaybolacaktır. Her actor aynı anda yalnızca bir mesajı işlediğinden thread'ler konusunda endişelenmesi gerekmez.

**Distributed actor frameworks** (Akka, Orleans ve Erlang/OTP gibi) bu programlama modelini bir uygulamayı birden fazla node'a ölçeklendirmek için kullanır. Gönderen ve alıcı aynı node'da mı yoksa farklı node'larda mı olursa olsun aynı mesaj geçirme mekanizması kullanılır. Farklı node'lardalarsa mesaj şeffaf biçimde bir bayt dizisine encode edilir, ağ üzerinden gönderilir ve diğer tarafta decode edilir.

**Location transparency**, actor modelinde RPC'ye kıyasla daha iyi çalışır; çünkü actor modeli zaten tek bir süreç içinde bile mesajların kaybolabileceğini varsayar. Ağ üzerindeki gecikme büyük olasılıkla aynı süreç içindekinden daha yüksek olsa da, actor modelini kullanırken yerel ile uzak iletişim arasındaki temel uyumsuzluk daha azdır.

Bir distributed actor framework özünde bir message broker ile actor programlama modelini tek bir framework içinde entegre eder. Ancak actor tabanlı uygulamanızın rolling upgrade'lerini gerçekleştirmek istiyorsanız, yine de forward ve backward compatibility konularında endişelenmeniz gerekir; çünkü mesajlar yeni sürümü çalıştıran bir node'dan eski sürümü çalıştıran bir node'a gönderilebilir ve bunun tersi de mümkündür. Bu, bu bölümde ele alınan encoding'lerden biri kullanılarak sağlanabilir.

<a id='16'></a>
## 16. Özet (Summary)

Bu bölümde veri yapılarını ağ üzerinde ya da diskte baytlara dönüştürmenin çeşitli yollarını inceledik. Bu encoding'lerin ayrıntılarının yalnızca verimliliği değil, daha da önemlisi uygulamaların mimarisini ve bunları geliştirme seçeneklerini nasıl etkilediğini gördük.

Pek çok servisin **rolling upgrade** (aşamalı güncelleme) desteklemesi gerekir; burada bir servisin yeni sürümü tüm node'lara aynı anda değil, yavaş yavaş birkaç node'a deploy edilir. Rolling upgrade'ler yeni sürümlerin servis kesintisi olmaksızın yayınlanmasına olanak tanır (bu nedenle nadir büyük sürümler yerine sık küçük sürümleri teşvik eder) ve deploy'ları daha az riskli hale getirir (hatalı sürümlerin çok sayıda kullanıcıyı etkilemeden önce tespit edilip geri alınmasına olanak tanır). Bu özellikler **evolvability** için son derece faydalıdır.

Rolling upgrade sırasında ya da çeşitli nedenlerle, sistemde farklı node'ların uygulamamızın farklı kod sürümlerini çalıştırdığını varsaymamız gerekir. Dolayısıyla sistemde akan tüm verilerin **backward compatibility** (yeni kod eski veriyi okuyabilir) ve **forward compatibility** (eski kod yeni veriyi okuyabilir) sağlayan biçimde encode edilmesi önemlidir.

### Encoding Formatları ve Uyumluluk Özellikleri

| Encoding Türü | Özellikler |
|---|---|
| **Language-specific encodings** | Tek bir programlama diliyle sınırlı; forward ve backward compatibility sağlamakta çoğunlukla başarısız |
| **Textual formats** (JSON, XML, CSV) | Yaygın; uyumlulukları nasıl kullandığınıza bağlı. Opsiyonel schema dilleri var. Sayılar ve binary string'ler konusunda dikkatli olmak gerekiyor |
| **Binary schema-driven formats** (Protocol Buffers, Avro) | Kompakt, verimli encoding; açıkça tanımlanmış forward/backward compatibility semantiği. Statik tipli dillerde dokümantasyon ve code generation için faydalı. Ancak decode edilmeden insan tarafından okunamaz |

### Veri Akışı Modları

Veri encoding'lerinin önemli olduğu çeşitli veri akışı modlarını da ele aldık:

| Mod | Encoding Akışı |
|---|---|
| **Databases** | Veritabanına yazan süreç veriyi encode eder, veritabanından okuyan süreç decode eder |
| **RPC and REST APIs** | Client isteği encode eder; server isteği decode edip yanıtı encode eder; client yanıtı decode eder |
| **Event-driven architectures** (message brokers veya actors kullanarak) | Node'lar birbirlerine gönderen tarafından encode edilip alıcı tarafından decode edilen mesajlar göndererek iletişim kurar |

Biraz dikkat göstererek backward/forward compatibility ve rolling upgrade'lerin oldukça başarılabilir olduğu sonucuna varabiliriz.

> *Uygulamanızın evrimi hızlı, deploy'larınız sık olsun!*

In [2]:
# Bölüm 5'teki kavramları özetleyen bir referans tablosu
concepts = {
    "Encoding (Serialization, Marshaling)": "Bellek içi gösterimden bayt dizisine dönüşüm",
    "Decoding (Deserialization, Unmarshaling)": "Bayt dizisinden bellek içi gösterime dönüşüm",
    "Backward Compatibility": "Yeni kod eski veriyi okuyabilir",
    "Forward Compatibility": "Eski kod yeni veriyi okuyabilir",
    "Rolling Upgrade": "Yeni sürümün hizmet kesintisi olmaksızın aşamalı deploy edilmesi",
    "Schema Evolution": "Schema'ların zamanla backward/forward uyumlu değişimi",
    "Protocol Buffers Field Tags": "Alanları ad yerine sayıyla tanımlar; kompakt binary encoding sağlar",
    "Avro Writer/Reader Schema": "Writer ve reader farklı schema sürümleri kullanabilir; Avro farkları çözer",
    "Message Broker": "Event-driven sistemlerde mesajları geçici olarak depolayan aracı",
    "Durable Execution": "Workflow adımları için exactly-once semantiği sağlar",
    "Actor Model": "Concurrency için; aktörler asenkron mesajlarla iletişim kurar",
    "Service Discovery": "Client'ın bağlanmak için servisin adresini bulması sorunu",
    "Load Balancing": "İstekleri birden fazla servis instance'ına dağıtma",
    "Idempotence": "Bir operasyonu birden fazla kez yapmak tek seferlik yapmakla aynı sonucu üretir",
    "Data Outlives Code": "Veritabanı kaydı, onu yazan kod sürümünden çok daha uzun yaşar",
}

print("📚 Bölüm 5 — Encoding and Evolution: Temel Kavramlar\n")
print("-" * 70)
for term, definition in concepts.items():
    print(f"\n🔑 {term}")
    print(f"   → {definition}")

📚 Bölüm 5 — Encoding and Evolution: Temel Kavramlar

----------------------------------------------------------------------

🔑 Encoding (Serialization, Marshaling)
   → Bellek içi gösterimden bayt dizisine dönüşüm

🔑 Decoding (Deserialization, Unmarshaling)
   → Bayt dizisinden bellek içi gösterime dönüşüm

🔑 Backward Compatibility
   → Yeni kod eski veriyi okuyabilir

🔑 Forward Compatibility
   → Eski kod yeni veriyi okuyabilir

🔑 Rolling Upgrade
   → Yeni sürümün hizmet kesintisi olmaksızın aşamalı deploy edilmesi

🔑 Schema Evolution
   → Schema'ların zamanla backward/forward uyumlu değişimi

🔑 Protocol Buffers Field Tags
   → Alanları ad yerine sayıyla tanımlar; kompakt binary encoding sağlar

🔑 Avro Writer/Reader Schema
   → Writer ve reader farklı schema sürümleri kullanabilir; Avro farkları çözer

🔑 Message Broker
   → Event-driven sistemlerde mesajları geçici olarak depolayan aracı

🔑 Durable Execution
   → Workflow adımları için exactly-once semantiği sağlar

🔑 Actor Model
   →